In [1]:
import numpy as np
import pandas as pd
import os
from collections import defaultdict

def calculate_kpi_metrics(df):
    if len(df) == 0:
        return 0, 0, 0
    trips = len(df)
    revenue = df["total_amount"].sum()
    speed_median = df["speed_mph"].median() if "speed_mph" in df.columns else 0
    return trips, revenue, speed_median

def clean_month(month, soft_percentiles=True):
    file = f"../../raw/yellow_tripdata_2021-{month:02d}.parquet"
    print(f"--- Đang xử lý tháng {month:02d}: {file} ---")
    
    # TỐI ƯU: Chỉ load các cột thực sự dùng để tiết kiệm RAM
    cols_to_use = [
        "VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime", "passenger_count",
        "trip_distance", "RatecodeID", "store_and_fwd_flag", "PULocationID", "DOLocationID",
        "payment_type", "fare_amount", "extra", "mta_tax", "tip_amount", "tolls_amount",
        "improvement_surcharge", "total_amount", "congestion_surcharge", "airport_fee"
    ]
    df = pd.read_parquet(file, columns=cols_to_use)

    # 1. Chuển hoá datetime
    df["tpep_pickup_datetime"]  = pd.to_datetime(df["tpep_pickup_datetime"], errors="coerce")
    df["tpep_dropoff_datetime"] = pd.to_datetime(df["tpep_dropoff_datetime"], errors="coerce")

    # 2. Chuẩn hoá numeric
    numeric_cols = [
        "passenger_count", "trip_distance", "fare_amount", "extra", "mta_tax",
        "tip_amount", "tolls_amount", "improvement_surcharge", "total_amount",
        "congestion_surcharge", "airport_fee", "RatecodeID"
    ]
    for c in numeric_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # 3. Tính toán Trip duration
    df["trip_duration"] = (df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]).dt.total_seconds() / 60

    # 4. Tách Flex Fare trips (Các chuyến không đo bằng đồng hồ điện tử)
    df["is_flex_fare"] = (df["payment_type"] == 0)
    metered = df[~df["is_flex_fare"]].copy()
    flex    = df[df["is_flex_fare"]].copy()

    total_rows = len(metered)

    # 5. Thiết lập Hard QA
    lookup = pd.read_csv("../../raw/taxi_zone_lookup.csv")
    valid_ids = lookup["LocationID"].unique()

    metered["valid_vendor"] = metered["VendorID"].isin([1, 2, 6, 7])
    metered["valid_time"]   = (metered["tpep_dropoff_datetime"] >= metered["tpep_pickup_datetime"]) & \
                              (metered["trip_duration"] > 0) & (metered["tpep_pickup_datetime"].dt.year == 2021)
    metered["valid_passenger"] = metered["passenger_count"].isin([1, 2, 3, 4, 5, 6, 99])
    metered["valid_ratecode"]  = metered["RatecodeID"].isin([1, 2, 3, 4, 5, 6])
    metered["valid_store"]     = metered["store_and_fwd_flag"].isin(["Y", "N"])
    metered["valid_payment"]   = metered["payment_type"].isin([1, 2, 3, 4, 5, 6])
    metered["valid_PULocationID"] = metered["PULocationID"].isin(valid_ids)
    metered["valid_DOLocationID"] = metered["DOLocationID"].isin(valid_ids)

    # 6. Tính tốc độ (mph) và giới hạn vật lý
    metered["speed_mph"] = metered["trip_distance"] / ((metered["trip_duration"] / 60) + 0.001)
    metered["valid_speed"] = (metered["speed_mph"] >= 1) & (metered["speed_mph"] <= metered["speed_mph"].quantile(0.999))

    # 7. Soft QA (Xử lý Outlier bằng phân vị)
    if soft_percentiles:
        for col in ["trip_distance", "trip_duration", "fare_amount"]:
            lower = metered[col].quantile(0.0001)
            upper = metered[col].quantile(0.9999)
            metered[f"valid_{col}"] = (metered[col] >= lower) & (metered[col] <= upper)
        
        price_per_mile = metered["fare_amount"] / metered["trip_distance"].replace(0, 0.001)
        metered["valid_economics"] = ~((metered["trip_distance"] > 5) & (price_per_mile < 0.5))

    # 8. Kiểm toán tài chính tài xế (Hard QA phụ)
    metered["valid_tip"] = ((metered["payment_type"] == 1) & (metered["tip_amount"] >= 0)) | \
                            ((metered["payment_type"] != 1) & (metered["tip_amount"] == 0))
    metered["valid_extra"] = metered["extra"] >= 0
    metered["valid_mta"]   = metered["mta_tax"].isin([0, 0.5])
    metered["valid_improvement"] = metered["improvement_surcharge"].isin([0, 0.3])
    metered["valid_congestion"]  = metered["congestion_surcharge"].isin([0, 0.75, 2.5])
    metered["valid_tolls"] = metered["tolls_amount"] >= 0
    metered["valid_airport"] = metered["airport_fee"].fillna(0) >= 0

    # 9. Tổng hợp cờ kiểm định
    flag_cols = [c for c in metered.columns if c.startswith("valid_")]
    metered["valid_all"] = metered[flag_cols].all(axis=1)

    # ĐÚNG YÊU CẦU: Tạo bảng tổng hợp lỗi QA của tháng
    qa_summary = []
    for col in flag_cols:
        fail_cnt = (~metered[col]).sum()
        qa_summary.append({
            "rule": col,
            "fail_count": fail_cnt
        })
    qa_summary_df = pd.DataFrame(qa_summary)

    # Tách tập dữ liệu Sạch và Lỗi
    clean = metered[metered["valid_all"]].copy()
    bad   = metered[~metered["valid_all"]].copy()

    # PHÂN TÍCH TÁC ĐỘNG QA LÊN KPI
    raw_trips, raw_rev, raw_speed = calculate_kpi_metrics(metered)
    clean_trips, clean_rev, clean_speed = calculate_kpi_metrics(clean)

    impact_data = {
        "month": month,
        "raw_revenue": raw_rev,
        "clean_revenue": clean_rev,
        "revenue_risk": raw_rev - clean_rev,
        "raw_speed_p50": raw_speed,
        "clean_speed_p50": clean_speed,
        "speed_diff": raw_speed - clean_speed
    }

    # In kết quả kiểm tra nhanh của tháng ra màn hình log
    print(f"Kết quả tháng {month:02d} -> Tổng: {total_rows:,} | Sạch: {len(clean):,} | Lỗi: {len(bad):,} | Flex: {len(flex):,}")

    # Ghi file ra ổ cứng
    os.makedirs("../../processed/clean_data", exist_ok=True)
    os.makedirs("../../processed/bad_data", exist_ok=True)
    os.makedirs("../../processed/flex_fare", exist_ok=True)

    clean.to_parquet(f"../../processed/clean_data/clean_2021-{month:02d}.parquet", index=False)
    bad.to_parquet(f"../../processed/bad_data/bad_2021-{month:02d}.parquet", index=False)
    flex.to_parquet(f"../../processed/flex_fare/flex_2021-{month:02d}.parquet", index=False)

    return len(clean), len(bad), len(flex), qa_summary_df, total_rows, impact_data

# =====================================================================
# TIẾN TRÌNH TỔNG HỢP VÀ CHẠY TOÀN VÒNG ĐỜI DỮ LIỆU 12 THÁNG (2021)
# =====================================================================
total_clean = 0
total_bad   = 0
total_flex  = 0
total_rows_all = 0
rule_fail_all = defaultdict(int)
impact_results = []

for month in range(1, 13):
    clean_cnt, bad_cnt, flex_cnt, qa_df, total_rows, impact_data = clean_month(month)
    
    impact_results.append(impact_data)
    total_clean += clean_cnt
    total_bad   += bad_cnt
    total_flex  += flex_cnt
    total_rows_all += total_rows

    for _, row in qa_df.iterrows():
        rule_fail_all[row["rule"]] += row["fail_count"]

# Xuất báo cáo tổng quan quy luật lỗi QA của cả năm
qa_year = pd.DataFrame([
    {
        "rule": rule,
        "fail_count": cnt,
        "fail_rate_percent": round(cnt / total_rows_all * 100, 4)
    }
    for rule, cnt in rule_fail_all.items()
]).sort_values("fail_rate_percent", ascending=False)

print("\n" + "="*50)
print("QA SUMMARY - FULL YEAR (2021)")
print("" + "="*50)
print(qa_year.to_string(index=False))

print("\n" + "="*50)
print("THỐNG KÊ SẢN LƯỢNG DÒNG DỮ LIỆU")
print("" + "="*50)
print(f"Tổng dòng clean (metered) 12 tháng: {total_clean:,}")
print(f"Tổng dòng bad (metered) 12 tháng:   {total_bad:,}")
print(f"Tổng chuyến Flex Fare 12 tháng:    {total_flex:,}")

# In báo cáo phân tích tác động trực tiếp phục vụ viết Word
print("\n" + "="*50)
print("=== PHÂN TÍCH TÁC ĐỘNG QA LÊN KPI (QA IMPACT) ===")
print("" + "="*50)
impact_df = pd.DataFrame(impact_results)
total_risk = impact_df['revenue_risk'].sum()
print(f"Tổng doanh thu rủi ro (Nhiễu toán học) đã loại bỏ cả năm: ${total_risk:,.2f}\n")
print(impact_df[["month", "revenue_risk", "raw_speed_p50", "clean_speed_p50"]].to_string(index=False))

# Lưu file kết quả phục vụ vẽ biểu đồ và nạp vào Trợ lý ảo
impact_df.to_csv("qa_impact_analysis.csv", index=False)
print("\n[HỆ THỐNG]: Đã xuất thành công file 'qa_impact_analysis.csv'. Ready cho cấu phần học máy tiếp theo.")

--- Đang xử lý tháng 01: ../../raw/yellow_tripdata_2021-01.parquet ---
Kết quả tháng 01 -> Tổng: 1,271,417 | Sạch: 1,216,239 | Lỗi: 55,178 | Flex: 98,352
--- Đang xử lý tháng 02: ../../raw/yellow_tripdata_2021-02.parquet ---
Kết quả tháng 02 -> Tổng: 1,273,463 | Sạch: 1,220,203 | Lỗi: 53,260 | Flex: 98,246
--- Đang xử lý tháng 03: ../../raw/yellow_tripdata_2021-03.parquet ---
Kết quả tháng 03 -> Tổng: 1,797,232 | Sạch: 1,721,660 | Lỗi: 75,572 | Flex: 127,920
--- Đang xử lý tháng 04: ../../raw/yellow_tripdata_2021-04.parquet ---
Kết quả tháng 04 -> Tổng: 2,043,167 | Sạch: 1,956,853 | Lỗi: 86,314 | Flex: 128,020
--- Đang xử lý tháng 05: ../../raw/yellow_tripdata_2021-05.parquet ---
Kết quả tháng 05 -> Tổng: 2,379,813 | Sạch: 2,283,477 | Lỗi: 96,336 | Flex: 127,296
--- Đang xử lý tháng 06: ../../raw/yellow_tripdata_2021-06.parquet ---
Kết quả tháng 06 -> Tổng: 2,710,726 | Sạch: 2,594,646 | Lỗi: 116,080 | Flex: 123,538
--- Đang xử lý tháng 07: ../../raw/yellow_tripdata_2021-07.parquet ---
